In [3]:
import duckdb
import pandas as pd
import time


data_path = "floodnet_full_dataset_merged_with_weather.parquet"
con = duckdb.connect()
output_path = "delineated_storms.parquet"

print("Step 1: Executing out-of-core storm delineation...")

# Instead of .df(), we use 'COPY ... TO ... (FORMAT PARQUET)'
# This keeps the memory footprint extremely low.
query = f"""
COPY (
    WITH rain_analysis AS (
        SELECT *,
               SUM("precip_1hr [inch]") OVER (
                   PARTITION BY deployment_id
                   ORDER BY time
                   ROWS BETWEEN 60 PRECEDING AND CURRENT ROW
               ) as rolling_hour_precip
        FROM '{data_path}'
    ),
    storm_triggers AS (
        SELECT *,
               CASE WHEN rolling_hour_precip >= 0.1 THEN 1 ELSE 0 END as is_storm_core
        FROM rain_analysis
    ),
    buffered_events AS (
        SELECT *,
               MAX(is_storm_core) OVER (
                   PARTITION BY deployment_id
                   ORDER BY time
                   ROWS BETWEEN 1440 PRECEDING AND 1440 FOLLOWING
               ) as in_storm_window
        FROM storm_triggers
    )
    SELECT * EXCLUDE (rolling_hour_precip, is_storm_core, in_storm_window)
    FROM buffered_events
    WHERE in_storm_window = 1
) TO '{output_path}' (FORMAT PARQUET);
"""

con.execute(query)
print(f"✅ Success! Delineated data saved to disk: {output_path}")

Step 1: Executing out-of-core storm delineation...
✅ Success! Delineated data saved to disk: delineated_storms.parquet


In [2]:
import duckdb

con = duckdb.connect()
# Use the output path we defined for the delineated storms
output_path = "delineated_storms.parquet"

# Query the metadata
column_info = con.execute(f"DESCRIBE SELECT * FROM '{output_path}' LIMIT 0").df()

print(f"📊 Total Columns Saved: {len(column_info)}")
print("-" * 30)
print(column_info['column_name'].tolist())

📊 Total Columns Saved: 38
------------------------------
['deployment_id', 'time', 'depth_proc_mm', 'depth_inches', 'name', 'date_deployed', 'date_down', 'deploy_type', 'location', 'image', 'sensor_mount', 'mounted_over', 'sensor_status', 'coordinates', 'borough', 'intersection', 'latitude', 'longitude', 'STATION', 'dist_to_station_ft', 'DATE', 'elevation [feet]', 'temp_2m [degF]', 'relative_humidity [percent]', 'dewpoint [degF]', 'precip_incremental [inch]', 'precip_max_intensity [inch/hour]', 'precip_1hr [inch]', 'frozen_soil_05cm [bit]', 'frozen_soil_25cm [bit]', 'frozen_soil_50cm [bit]', 'soil_temp_05cm [degF]', 'soil_temp_25cm [degF]', 'soil_temp_50cm [degF]', 'soil_moisture_05cm [m^3/m^3]', 'soil_moisture_25cm [m^3/m^3]', 'soil_moisture_50cm [m^3/m^3]', 'snow_depth [inch]']


In [ ]:
import pandas as pd

print("Loading delineated storms into memory...")
# Load only necessary columns for the "Model Shootout"
df_storms = pd.read_parquet(output_path)

# THE CRITICAL MEMORY STEP: Downcast floats to save 50% RAM
for col in df_storms.select_dtypes(include=['float64']).columns:
    df_storms[col] = df_storms[col].astype('float32')

print(f"Data Loaded. Final Memory Usage: {df_storms.memory_usage().sum() / 1024**2:.2f} MB")

Loading delineated storms into memory...


In [ ]:
import optuna
from sklearn.linear_model import LinearRegression
import numpy as np

def calculate_nse(y_true, y_pred):
    """Hydroinformatics standard: Nash-Sutcliffe Efficiency"""
    return 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))

def objective(trial):
    # AUTOMATED PARAMETER TUNING
    # Example for Logarithmic Regression Baseline
    shift_value = trial.suggest_float("log_shift", 0.001, 0.1) # Handle log(0)
    
    # Prep features (Precip, Soil Moisture, etc.)
    X = df_storms[['precip_1hr [inch]', 'soil_moisture_05cm [m^3/m^3]']].fillna(0)
    y = np.log(df_storms['depth_inches'] + shift_value)
    
    # Train/Test Split
    model = LinearRegression().fit(X, y)
    preds = np.exp(model.predict(X)) - shift_value
    
    # Maximize NSE
    score = calculate_nse(df_storms['depth_inches'], preds)
    return score

# Create the study to maximize the best-performing version
study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=20)